# ContextQA - Phase 1: SQuAD v1.1 Dataset Exploration

Welcome to **ContextQA**! In this notebook, we explore the Stanford Question Answering Dataset (**SQuAD v1.1**) from first principles.

### Objectives:
1. Verify the Python environment, PyTorch, and CUDA GPU acceleration (`RTX 3060`).
2. Load SQuAD v1.1 using Hugging Face Datasets with local caching.
3. Inspect dataset splits, schema, and sample questions/contexts.
4. Analyze the distributions of context lengths, question lengths, and answer lengths.
5. Understand why tokenization and sliding windows are mathematically necessary for Transformer architectures.

In [ ]:
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device:      {torch.cuda.get_device_name(0)}")
    print(f"VRAM Total:      {torch.cuda.get_device_properties(0).total_memory / (1024**2):.0f} MB")

## 1. Load SQuAD v1.1 Dataset

In [ ]:
from datasets import load_dataset

# Load SQuAD v1.1 from the Hugging Face Hub (cached automatically)
dataset = load_dataset("rajpurkar/squad")
print(dataset)
print(f"\nTrain rows:      {len(dataset['train']):,}")
print(f"Validation rows: {len(dataset['validation']):,}")

## 2. Inspect Dataset Schema & Sample Example

In [ ]:
example = dataset["train"][0]
print("Sample ID:", example["id"])
print("Title:", example["title"])
print("\nContext:\n", example["context"])
print("\nQuestion:\n", example["question"])
print("\nAnswers:\n", example["answers"])

# Verify character offset alignment
start_char = example["answers"]["answer_start"][0]
ans_text = example["answers"]["text"][0]
extracted_text = example["context"][start_char : start_char + len(ans_text)]
print(f"\nCharacter slice verification: '{extracted_text}' == '{ans_text}' -> {extracted_text == ans_text}")

## 3. Compute Dataset Statistics & Length Distributions

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

train_data = dataset["train"]
records = []
for row in train_data:
    ans_text = row["answers"]["text"][0] if row["answers"]["text"] else ""
    records.append({
        "context_words": len(row["context"].split()),
        "question_words": len(row["question"].split()),
        "answer_words": len(ans_text.split()),
        "answer_start": row["answers"]["answer_start"][0] if row["answers"]["answer_start"] else -1,
    })

df = pd.DataFrame(records)
df.describe(percentiles=[0.25, 0.50, 0.75, 0.95, 0.99])

## 4. Key Takeaways & Architecture Insights for Phase 2

1. **Context Length**: The 95th percentile context length is 213 words (~280 subword tokens). Thus, a `max_seq_length = 384` covers the vast majority of passages in a single pass.
2. **Sliding Windows**: Passages longer than 384 tokens require overlapping windows (`doc_stride = 128`) to prevent answers near chunk boundaries from being truncated.
3. **Answer Length**: The median answer length is only 2 words, and 95% of answers have 10 words or fewer. A `max_answer_length = 30` tokens is ideal for filtering out implausibly long spans during inference.